# Чувствительность политики к $\eta$ и $\gamma$

Ноутбук **не обучает** модели: он читает готовые `report.json` и чекпоинты из папки сетки. Для стационарной задачи $\gamma=\bar\gamma$ — коэффициент энтропийного риска **на единицу кредитной книги**; для старой конечногоризонтной сетки $\gamma$ имеет другую нормировку. Эти два типа запусков не смешиваются.

Главное сравнение проводится в **одних и тех же состояниях** $(r,F,Y,E)$ для всех моделей. $f$ — доля fixed **в новых выдачах**, не $F/L$ в действующем балансе. Значение $V$ при разных $\gamma$ не следует сравнивать как один и тот же критерий: предпочтения различны. $\Delta V$ ниже сравнивается только как диагностика чувствительности соответствующего функционала, а не как рыночная цена.

In [ ]:
from pathlib import Path
from collections import defaultdict
import json, os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt

# Укажите папку вашей сетки, например 'runs/stationary_eta_gamma_sweep'.
# None: автоматически выбрать наиболее свежую сетку с меняющимися eta и gamma;
# стационарные сетки имеют приоритет над старыми конечногоризонтными.
SWEEP_DIR = None
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ACTION_GRID = 129              # точность сравнения политик
MAP_GRID = 15 if os.environ.get('IRRBB_NOTEBOOK_SMOKE') else 31
SCENARIO_SHOCK = 0.01
RATES = np.linspace(.02, .18, MAP_GRID)
FIXED_SHARES = np.linspace(.1, .9, MAP_GRID)

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'continuous_time_irrbb/risk_sensitive/stationary_solver.py').exists()), None)
if root is None:
    raise FileNotFoundError('Запустите Jupyter внутри проекта IRRBB')
risk_dir = root / 'continuous_time_irrbb/risk_sensitive'
sys.path.insert(0, str(root))
from continuous_time_irrbb.risk_sensitive.stationary_solver import StationarySolution
from continuous_time_irrbb.risk_sensitive.solver import FittedSolution

def read_point(path):
    try:
        d = json.loads(path.read_text(encoding='utf-8'))
    except (OSError, ValueError):
        return None
    if 'stationary_config' in d and (path.parent/'stationary_solution.pt').exists():
        mode, cfg, filename = 'stationary', d['stationary_config'], 'stationary_solution.pt'
        gamma = cfg['risk_aversion_per_loan']
    elif 'solver_config' in d and (path.parent/'solution.pt').exists():
        mode, cfg, filename = 'finite', d['solver_config'], 'solution.pt'
        gamma = cfg['risk_aversion']
    else:
        return None
    p = d['parameters']
    return dict(path=path, checkpoint=path.parent/filename, run=path.parent.name,
                mode=mode, gamma=float(gamma), eta=float(p['floating_demand_slope']),
                parameters=p, config=cfg, report=d)

def find_sweep():
    candidates = defaultdict(list)
    for path in (risk_dir/'runs').rglob('report.json'):
        point = read_point(path)
        if point:
            candidates[(path.parent.parent, point['mode'])].append(point)
    usable = []
    for (folder, mode), points in candidates.items():
        if len({q['gamma'] for q in points}) >= 2 and len({q['eta'] for q in points}) >= 2:
            usable.append((mode == 'stationary', folder.stat().st_mtime, len(points), folder))
    if not usable:
        raise FileNotFoundError('Не найдена полная сетка eta × gamma. Укажите SWEEP_DIR вручную.')
    return max(usable)[-1]

sweep_dir = find_sweep() if SWEEP_DIR is None else Path(SWEEP_DIR).expanduser()
if not sweep_dir.is_absolute(): sweep_dir = risk_dir / sweep_dir
if not sweep_dir.is_dir(): raise FileNotFoundError(sweep_dir)
points = [q for path in sorted(sweep_dir.rglob('report.json')) if (q := read_point(path))]
if not points: raise FileNotFoundError(f'В {sweep_dir} нет готовых чекпоинтов с report.json')
modes = {q['mode'] for q in points}
if len(modes) != 1: raise ValueError(f'В папке смешаны разные постановки: {modes}')
mode = modes.pop()
print(f'Папка: {sweep_dir}\nПостановка: {mode}; моделей: {len(points)}; device: {DEVICE}')

## Полнота сетки и сопоставимость

Меняться должны только $\eta$ и $\gamma$. Если другие экономические параметры тоже различаются, их влияние нельзя отделить по этим графикам. Отчётные `converged` и невязка — **диагностика**, а не доказательство оптимальности; сравнение небольших изменений действия особенно чувствительно к численной ошибке.

In [ ]:
gammas = sorted({q['gamma'] for q in points})
etas = sorted({q['eta'] for q in points})
by_pair = defaultdict(list)
for q in points: by_pair[(q['gamma'],q['eta'])].append(q)
duplicates = {key:[q['run'] for q in group] for key,group in by_pair.items() if len(group)>1}
if duplicates: raise ValueError(f'Повторные точки (gamma, eta), выберите одну сетку: {duplicates}')
missing = [(g,e) for g in gammas for e in etas if (g,e) not in by_pair]
print(f'γ: {gammas}\nη, б.п.: {[round(e*10000,2) for e in etas]}')
print(f'Заполнено {len(by_pair)}/{len(gammas)*len(etas)} точек; отсутствуют: {missing or "нет"}')
reference_p = points[0]['parameters']
other_changes = {}
for key in reference_p:
    if key == 'floating_demand_slope': continue
    values = {str(q['parameters'].get(key)) for q in points}
    if len(values)>1: other_changes[key]=sorted(values)
print('Другие меняющиеся параметры:', other_changes or 'нет')
if other_changes: print('ВНИМАНИЕ: эффекты eta/gamma могут быть смешаны с изменением других параметров.')
if mode == 'stationary':
    for q in points:
        d=q['report'];q['residual']=float(d['diagnostics'][-1]['bellman_rmse'])
        q['quality']=bool(d.get('converged_on_validation_cloud'))
else:
    for q in points:
        d=q['report'];q['residual']=abs(float(d.get('reference_bellman_residual_0',np.nan)))
        q['quality']=not bool(d.get('quality_flags'))
print(f'Контроль качества: {sum(q["quality"] for q in points)}/{len(points)} без флага проблемы.')
for q in points:
    if not q['quality']:
        print(f'  {q["run"]}: γ={q["gamma"]:g}, η={q["eta"]*1e4:g} б.п., '
              f'невязка={q["residual"]:.4g}')

## Сравнение в одном и том же банке

Опорное состояние: $r=8\%$, $F/L=50\%$, $E/L=12\%$, старый fixed-купон $10\%$, старый float-спред $1.25\%$. Поэтому $Y=0.10F+0.0125(L-F)$. Для усреднения политики ниже используется **равномерная диагностическая сетка состояний**, а не распределение, возникающее при каждой политике. Это позволяет сравнить именно правила принятия решений, без подмены эффекта параметра изменением состава посещённых состояний.

In [ ]:
RATE0, FIXED0, CAR0 = .08, .50, .12
OLD_FIXED_COUPON, OLD_FLOAT_SPREAD = .10, .0125
state_r = np.array([.04,.08,.12,.16])
state_f = np.array([.2,.5,.8])

def build_states(loans, rates, fixed_shares, car=CAR0):
    rr, aa = np.meshgrid(np.asarray(rates),np.asarray(fixed_shares),indexing='ij')
    F = aa*loans
    Y = OLD_FIXED_COUPON*F + OLD_FLOAT_SPREAD*(loans-F)
    E = np.full_like(rr,car*loans)
    x = np.stack([rr,F,Y,E],axis=-1)
    return torch.as_tensor(x.reshape(-1,4),device=DEVICE,dtype=torch.float64)

def load_model(q):
    cls = StationarySolution if mode=='stationary' else FittedSolution
    return cls.load(q['checkpoint'],device=DEVICE,dtype=torch.float64)

def model_action(solution,x):
    with torch.inference_mode():
        return torch.cat([solution.action(part,action_count=ACTION_GRID)
                          if mode=='stationary' else solution.action(0,part,action_count=ACTION_GRID)
                          for part in x.split(128)]).cpu().numpy()

def model_value(solution,x):
    with torch.inference_mode():
        return (solution.value(x) if mode=='stationary' else solution.value(0,x)).cpu().numpy()

for q in points:
    model=load_model(q)
    L=model.environment.p.loans
    x0=build_states(L,[RATE0],[FIXED0])
    x1=x0.clone();x1[:,0]+=SCENARIO_SHOCK
    cloud=build_states(L,state_r,state_f)
    q['f_reference']=float(model_action(model,x0)[0])
    acts=model_action(model,cloud)
    q['f_mean_common_grid']=float(acts.mean())
    q['interior_fraction']=float(((acts>.05)&(acts<.95)).mean())
    q['delta_v_reference']=float(model_value(model,x1)[0]-model_value(model,x0)[0])
    del model
    if DEVICE=='cuda': torch.cuda.empty_cache()

print('Политики пересчитаны в одинаковых состояниях; ΔV — шок +100 б.п. в опорном состоянии.')

In [ ]:
def matrix(key):
    values=np.full((len(gammas),len(etas)),np.nan)
    for i,g in enumerate(gammas):
        for j,e in enumerate(etas):
            if (g,e) in by_pair: values[i,j]=by_pair[(g,e)][0][key]
    return values

def heatmap(ax,values,title,fmt='.2f',cmap='viridis',vmin=None,vmax=None):
    im=ax.imshow(np.ma.masked_invalid(values),origin='lower',aspect='auto',
                 cmap=cmap,vmin=vmin,vmax=vmax)
    ax.set_xticks(range(len(etas)),[f'{e*1e4:g}' for e in etas])
    ax.set_yticks(range(len(gammas)),[f'{g:g}' for g in gammas])
    ax.set(xlabel='η, б.п. на единицу доли float',ylabel='γ',title=title)
    for i in range(len(gammas)):
        for j in range(len(etas)):
            if np.isfinite(values[i,j]): ax.text(j,i,format(values[i,j],fmt),
                ha='center',va='center',fontsize=8,color='white' if values[i,j]<.5 else 'black')
    plt.colorbar(im,ax=ax)

fig,axes=plt.subplots(1,3,figsize=(17,4.8),constrained_layout=True)
heatmap(axes[0],matrix('f_reference'),'f в опорном состоянии',vmin=0,vmax=1)
heatmap(axes[1],matrix('f_mean_common_grid'),'Среднее f на общей сетке состояний',vmin=0,vmax=1)
heatmap(axes[2],matrix('interior_fraction'),'Доля неугловых решений на общей сетке',vmin=0,vmax=1)
plt.show()

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4.3),constrained_layout=True)
for i,g in enumerate(gammas):
    axes[0].plot(np.array(etas)*1e4,matrix('f_reference')[i],marker='o',label=f'γ={g:g}')
for j,e in enumerate(etas):
    axes[1].plot(gammas,matrix('f_reference')[:,j],marker='o',label=f'η={e*1e4:g} б.п.')
axes[0].set(xlabel='η, б.п. на единицу доли float',ylabel='f в опорном состоянии',
            title='Чувствительность к η',ylim=(-.03,1.03))
axes[1].set(xlabel='γ',ylabel='f в опорном состоянии',
            title='Чувствительность к γ',ylim=(-.03,1.03))
for ax in axes: ax.grid(alpha=.3);ax.legend(fontsize=8)
plt.show()

## Чувствительность стоимости и качество решения

На одной и той же книге $\Delta V=V(r+100\,\mathrm{б.п.})-V(r)$ может зависеть от обоих параметров. Для сравнения слабых эффектов важна невязка Беллмана. У стационарной модели ниже показана RMSE на контрольном облаке, у старой конечногоризонтной — абсолютная невязка в опорной точке; эти диагностические величины **не сопоставимы между постановками**.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4.5),constrained_layout=True)
heatmap(axes[0],matrix('delta_v_reference'),'ΔV: +100 б.п., общая опорная книга',fmt='+.2f',cmap='RdBu_r')
heatmap(axes[1],matrix('residual'),
        'Проверочная Bellman RMSE' if mode=='stationary' else '|Bellman residual| в опорной точке',
        fmt='.3f',cmap='magma')
plt.show()
print('Флаг сходимости по отчёту (1=да):')
print(np.array([[int(by_pair[(g,e)][0]['quality']) if (g,e) in by_pair else -1
                 for e in etas] for g in gammas]))

## Как меняется всё правило управления

Четыре крайние комбинации показывают $f(r,F/L)$ при одинаковых $Y/F$, старом float-спреде и $E/L$. Это важнее одной опорной точки: чувствительность параметров может менять наклон политики, а не только её уровень. Каждая карта пересчитывается из соответствующего сохранённого чекпоинта. Для стационарной политики время не является аргументом; для конечногоризонтной показано $t=0$.

In [ ]:
corners=[]
for g,e in [(gammas[0],etas[0]),(gammas[0],etas[-1]),
            (gammas[-1],etas[0]),(gammas[-1],etas[-1])]:
    if (g,e) in by_pair and (g,e) not in corners: corners.append((g,e))
fig,axes=plt.subplots(2,2,figsize=(12,9),sharex=True,sharey=True,constrained_layout=True)
for ax in axes.flat: ax.set_visible(False)
for ax,(g,e) in zip(axes.flat,corners):
    ax.set_visible(True)
    q=by_pair[(g,e)][0];model=load_model(q)
    x=build_states(model.environment.p.loans,RATES,FIXED_SHARES)
    values=model_action(model,x).reshape(len(RATES),len(FIXED_SHARES))
    im=ax.imshow(values.T,origin='lower',aspect='auto',cmap='viridis',vmin=0,vmax=1,
                 extent=[RATES[0]*100,RATES[-1]*100,FIXED_SHARES[0]*100,FIXED_SHARES[-1]*100])
    ax.scatter([RATE0*100],[FIXED0*100],s=30,c='white',edgecolors='black')
    ax.set(title=f'γ={g:g}, η={e*1e4:g} б.п.',xlabel='текущая ставка r, %',
           ylabel='fixed в книге F/L, %')
    del model
    if DEVICE=='cuda': torch.cuda.empty_cache()
fig.colorbar(im,ax=axes.ravel().tolist(),label='f: fixed в новых выдачах')
plt.show()

**Как читать результаты.** Рост $\eta$ сильнее снижает маржу float при большом объёме float-выдачи, поэтому обычно подталкивает к fixed. Рост $\gamma$ усиливает неприятие риска процентных потоков; направление изменения $f$ определяется текущей книгой и ставкой, а не является теоремой. Смотрите на карты, долю неугловых решений и невязку вместе. Если разница политик меньше шага сетки действий $1/(	exttt{ACTION\_GRID}-1)$ или меняется при небольшом уточнении решения, экономический вывод преждевременен.